# Longitudinal imaging features for dementia risk (simulation)

Compare age and MRI-derived volumes at the prediction visit with the same features plus annualized changes since the prior visit. Evaluate regularized logistic regression on subject-level out-of-fold probabilities. Generate 900 simulated trajectories per seed. Event risk depends on current level and latent progression rate. Use five stratified folds.

In [ ]:
from pathlib import Path
import subprocess, sys
root = Path.cwd() / 'longitudinal-imaging-landmark-heavy-run'
root.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install',*['numpy==2.2.6', 'scipy==1.15.3', 'scikit-learn==1.7.2', 'joblib==1.6.0']],check=True)

In [ ]:
(root / 'train.py').write_text('import json\nimport os\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    average_precision_score,\n    brier_score_loss,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ.get("PORTFOLIO_DATA", ROOT / "data"))\nSEEDS = [13, 37, 61]\n\n\ndef save(obj, name):\n    folder = ROOT / "checkpoints"\n    folder.mkdir(exist_ok=True)\n    joblib.dump(obj, folder / (name + ".joblib"), compress=3)\n\n\ndef scores(y, p):\n    return {\n        "auroc": float(roc_auc_score(y, p)),\n        "average_precision": float(average_precision_score(y, p)),\n    }\n\n\ndef dataset(seed):\n    from scipy.special import expit\n\n    rng = np.random.default_rng(seed)\n    n = 900\n    age = rng.normal(76, 7, n)\n    level = rng.normal(0.74, 0.045, n)\n    speed = rng.normal(-0.01, 0.022, n)\n    prev = level + rng.normal(0, 0.009, n)\n    latest = level + speed + rng.normal(0, 0.009, n)\n    etiv = rng.normal(1500, 140, n)\n    asf = 1700 / etiv\n    x = np.c_[\n        age,\n        latest,\n        etiv,\n        asf,\n        latest - prev,\n        rng.normal(0, 20, n),\n        rng.normal(0, 0.013, n),\n    ]\n    y = rng.binomial(1, expit(-1.5 - 12 * (latest - 0.74) - 36 * speed))\n    return x, y\n\n\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.model_selection import StratifiedKFold\n\n\ndef run(seed):\n    x, y = dataset(seed)\n    results = {}\n    folds = list(\n        StratifiedKFold(n_splits=5, shuffle=True, random_state=seed).split(x, y)\n    )\n    for name, k in [("latest_visit", 4), ("longitudinal", 7)]:\n        p = np.empty(len(y))\n        for i, (train, test) in enumerate(folds):\n            model = make_pipeline(\n                SimpleImputer(),\n                StandardScaler(),\n                LogisticRegression(C=0.08, max_iter=700),\n            ).fit(x[train, :k], y[train])\n            p[test] = model.predict_proba(x[test, :k])[:, 1]\n            save(model, f"{name}-{seed}-{i}")\n        results[name] = {\n            **scores(y, p),\n            "brier": float(brier_score_loss(y, p)),\n            "subjects": len(y),\n            "events": int(y.sum()),\n        }\n    return results\n\n\nif __name__ == "__main__":\n    runs = {s: run(s) for s in SEEDS}\n    first = next(iter(runs.values()))\n    means = {\n        k: {m: float(np.mean([r[k][m] for r in runs.values()])) for m in v}\n        for k, v in first.items()\n    }\n    (ROOT / "results.json").write_text(\n        json.dumps({"seeds": SEEDS, "runs": runs, "means": means}, indent=2)\n    )\n    print(means)\n',encoding='utf8')
subprocess.run([sys.executable,'train.py'],cwd=root,check=True)

In [ ]:
print((root/'results.json').read_text())